# NYC Mobility - Bronze Validation

## Why validate Bronze

Before moving to Silver, these checks confirm that Bronze received the required source records and that repeated ingestion did not duplicate batches. They report issues without fixing or removing rows.


## Green Taxi row count by source file

The recorded load evidence is March 44,208, April 44,238, and May 44,921. This consolidated check confirms those per-file counts in the current Bronze table.


In [0]:
-- validate bronze row counts by source file

SELECT
    source_file,
    COUNT(*) AS bronze_row_count
FROM `ftw-week-08`.`01_bronze`.`green_taxi`
GROUP BY source_file
ORDER BY source_file;


## Total Bronze rows

The verified final Green Taxi Bronze total is **133,367 rows**.


In [0]:
SELECT COUNT(*) AS total_bronze_rows
FROM `ftw-week-08`.`01_bronze`.`green_taxi`;

## Ingestion log

The recorded evidence shows one successful record for each March, April, and May source file, with row counts matching the loaded batches.


In [0]:
SELECT
    source_identifier,
    batch_id,
    status,
    rows_loaded,
    ingested_at
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`
WHERE source_name = 'green_taxi'
ORDER BY ingested_at;

This additional check makes duplicate log records visible per source identifier and batch.


In [0]:
-- validate one successful log record per monthly batch

SELECT
    source_identifier,
    batch_id,
    status,
    rows_loaded,
    COUNT(*) AS log_records
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`
WHERE source_name = 'green_taxi'
GROUP BY
    source_identifier,
    batch_id,
    status,
    rows_loaded
ORDER BY source_identifier;


The May-specific check records one log row.


In [0]:
SELECT COUNT(*) AS may_log_rows
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`
WHERE source_identifier = 'green_tripdata_2026-05.parquet';

## Idempotency evidence

We deliberately reran March and May in the Bronze load notebook. Both second runs inserted **0 rows**. March stayed at 44,208 rows, May did not create another batch, and the final total remained 133,367.


## Provenance

The provenance check requires every loaded Green Taxi row to carry its source file, batch ID, source system, and ingestion timestamp.


In [0]:
-- validate bronze provenance metadata

SELECT
    source_file,
    batch_id,
    source_system,
    COUNT(*) AS row_count,
    SUM(
        CASE
            WHEN source_file IS NULL
              OR batch_id IS NULL
              OR source_system IS NULL
              OR ingested_at IS NULL
            THEN 1
            ELSE 0
        END
    ) AS rows_with_missing_provenance
FROM `ftw-week-08`.`01_bronze`.`green_taxi`
GROUP BY
    source_file,
    batch_id,
    source_system
ORDER BY source_file;


## Source-to-Bronze reconciliation

This compares each verified landed Parquet count with its Bronze count. It does not modify either side.


In [0]:
-- reconcile each source file with bronze

WITH source_counts AS (
    SELECT
        'green_tripdata_2026-03.parquet' AS source_file,
        COUNT(*) AS source_row_count
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )

    UNION ALL

    SELECT
        'green_tripdata_2026-04.parquet' AS source_file,
        COUNT(*) AS source_row_count
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-04.parquet',
        format => 'parquet'
    )

    UNION ALL

    SELECT
        'green_tripdata_2026-05.parquet' AS source_file,
        COUNT(*) AS source_row_count
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-05.parquet',
        format => 'parquet'
    )
),

bronze_counts AS (
    SELECT
        source_file,
        COUNT(*) AS bronze_row_count
    FROM `ftw-week-08`.`01_bronze`.`green_taxi`
    GROUP BY source_file
)

SELECT
    source.source_file,
    source.source_row_count,
    COALESCE(bronze.bronze_row_count, 0) AS bronze_row_count,
    source.source_row_count = COALESCE(bronze.bronze_row_count, 0) AS counts_match
FROM source_counts AS source
LEFT JOIN bronze_counts AS bronze
    ON source.source_file = bronze.source_file
ORDER BY source.source_file;


## Taxi Zones

Taxi Zones passes when source and Bronze counts are both 265, required key nulls are 0, the duplicate query returns no rows, provenance is complete, and exactly one SUCCESS log receipt exists.


In [0]:
-- reconcile taxi zone source and bronze counts

WITH source_count AS (
    SELECT COUNT(*) AS row_count
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/taxi_zones/taxi_zone_lookup.csv',
        format => 'csv',
        header => true
    )
),
bronze_count AS (
    SELECT COUNT(*) AS row_count
    FROM `ftw-week-08`.`01_bronze`.`taxi_zones`
    WHERE source_file = 'taxi_zone_lookup.csv'
)
SELECT
    source_count.row_count AS source_row_count,
    bronze_count.row_count AS bronze_row_count,
    source_count.row_count = bronze_count.row_count AS counts_match
FROM source_count
CROSS JOIN bronze_count;


PASS criteria: `source_row_count = 265`, `bronze_row_count = 265`, and `counts_match = true`. This reports source-to-Bronze preservation without fixing any records.


In [0]:
-- report taxi zone null keys and missing provenance

SELECT
    SUM(CASE WHEN LocationID IS NULL THEN 1 ELSE 0 END) AS null_location_id,
    SUM(
        CASE
            WHEN source_system IS NULL
              OR source_file IS NULL
              OR batch_id IS NULL
              OR ingested_at IS NULL
            THEN 1 ELSE 0
        END
    ) AS rows_with_missing_provenance
FROM `ftw-week-08`.`01_bronze`.`taxi_zones`
WHERE source_file = 'taxi_zone_lookup.csv';


PASS criteria: `null_location_id = 0` and `rows_with_missing_provenance = 0`.


In [0]:
-- report duplicate taxi zone keys without removing them

SELECT
    LocationID,
    COUNT(*) AS record_count
FROM `ftw-week-08`.`01_bronze`.`taxi_zones`
WHERE source_file = 'taxi_zone_lookup.csv'
GROUP BY LocationID
HAVING COUNT(*) > 1;


PASS criterion: **0 result rows**, confirming that `LocationID` is unique.


In [0]:
-- validate one successful taxi zone ingestion receipt

SELECT
    source_identifier,
    batch_id,
    status,
    rows_loaded,
    COUNT(*) AS log_records
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`
WHERE source_name = 'taxi_zones'
  AND source_identifier = 'taxi_zone_lookup.csv'
  AND batch_id = 'taxi_zone_lookup.csv'
GROUP BY source_identifier, batch_id, status, rows_loaded;


PASS criteria: exactly one row with `status = SUCCESS`, `rows_loaded = 265`, and `log_records = 1`. The same-file rerun evidence is 0 inserted rows with the Bronze count remaining 265.


## Weather

Weather’s source inspection contains 2,208 hourly timestamps. Bronze has a different grain: one complete raw JSON file per batch.


In [0]:
-- validate the one-record weather raw batch and its provenance

SELECT
    COUNT(*) AS raw_batch_count,
    SUM(CASE WHEN raw_json IS NULL OR LENGTH(raw_json) = 0 THEN 1 ELSE 0 END) AS null_or_empty_raw_json,
    SUM(
        CASE
            WHEN source_system IS NULL
              OR source_url IS NULL
              OR source_file IS NULL
              OR batch_id IS NULL
              OR ingested_at IS NULL
            THEN 1 ELSE 0
        END
    ) AS rows_with_missing_provenance
FROM `ftw-week-08`.`01_bronze`.`weather_raw`
WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
  AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json';


PASS criteria: `raw_batch_count = 1`, `null_or_empty_raw_json = 0`, and `rows_with_missing_provenance = 0`. This count is not compared with 2,208 because those observations remain inside `raw_json`.


In [0]:
-- validate one successful weather ingestion receipt

SELECT
    source_identifier,
    batch_id,
    status,
    rows_loaded,
    COUNT(*) AS log_records
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`
WHERE source_name = 'weather'
  AND source_identifier = 'open_meteo_2026-03-01_2026-05-31.json'
  AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json'
GROUP BY source_identifier, batch_id, status, rows_loaded;


PASS criteria: exactly one row with `status = SUCCESS`, `rows_loaded = 1`, and `log_records = 1`. The same-file rerun evidence is 0 inserted rows with `raw_batch_count` remaining 1.


## Traffic Advisory — bonus

This validation covers one saved September 2026 scrape only. It does not claim analytical integration with March-May Taxi data.


In [0]:
-- validate one complete raw traffic scrape and its provenance

SELECT
    COUNT(*) AS raw_batch_count,
    SUM(CASE WHEN raw_html IS NULL OR LENGTH(raw_html) = 0 THEN 1 ELSE 0 END) AS null_or_empty_raw_html,
    SUM(CASE WHEN raw_metadata_json IS NULL OR LENGTH(raw_metadata_json) = 0 THEN 1 ELSE 0 END) AS null_or_empty_raw_metadata_json,
    SUM(
        CASE
            WHEN source_system IS NULL
              OR source_url IS NULL
              OR source_file IS NULL
              OR batch_id IS NULL
              OR ingested_at IS NULL
            THEN 1 ELSE 0
        END
    ) AS rows_with_missing_provenance
FROM `ftw-week-08`.`01_bronze`.`traffic_advisory_raw`
WHERE source_file = 'nyc_dot_weekend_traffic_20260914T040846Z.html'
  AND batch_id = '20260914T040846Z';


PASS criteria: `raw_batch_count = 1`, both payload checks are 0, and `rows_with_missing_provenance = 0`.


In [0]:
-- validate one successful traffic ingestion receipt

SELECT
    source_identifier,
    batch_id,
    status,
    rows_loaded,
    COUNT(*) AS log_records
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`
WHERE source_name = 'traffic_advisory'
  AND source_identifier = 'nyc_dot_weekend_traffic_20260914T040846Z.html'
  AND batch_id = '20260914T040846Z'
GROUP BY source_identifier, batch_id, status, rows_loaded;


PASS criteria: exactly one row with `status = SUCCESS`, `rows_loaded = 1`, and `log_records = 1`. The idempotency rerun uses the same saved HTML and metadata files and provides 0 inserted rows with the raw batch count remaining 1.


## Overall Bronze Quality Gate

This query computes the current status directly from the Databricks tables instead of hardcoding a result. The required gate contains Green Taxi, Taxi Zones, and Weather; Traffic Advisory is bonus and cannot fail the core gate.


In [0]:
-- compute the required core bronze gate from current table results

WITH checks AS (
    SELECT
        'green_taxi' AS source_name,
        COUNT(*) AS actual_rows,
        133367 AS expected_rows,
        SUM(
            CASE
                WHEN source_system IS NULL
                  OR source_file IS NULL
                  OR batch_id IS NULL
                  OR ingested_at IS NULL
                THEN 1 ELSE 0
            END
        ) AS rows_with_missing_provenance
    FROM `ftw-week-08`.`01_bronze`.`green_taxi`

    UNION ALL

    SELECT
        'taxi_zones' AS source_name,
        COUNT(*) AS actual_rows,
        265 AS expected_rows,
        SUM(
            CASE
                WHEN source_system IS NULL
                  OR source_file IS NULL
                  OR batch_id IS NULL
                  OR ingested_at IS NULL
                THEN 1 ELSE 0
            END
        ) AS rows_with_missing_provenance
    FROM `ftw-week-08`.`01_bronze`.`taxi_zones`
    WHERE source_file = 'taxi_zone_lookup.csv'

    UNION ALL

    SELECT
        'weather' AS source_name,
        COUNT(*) AS actual_rows,
        1 AS expected_rows,
        SUM(
            CASE
                WHEN raw_json IS NULL
                  OR source_system IS NULL
                  OR source_url IS NULL
                  OR source_file IS NULL
                  OR batch_id IS NULL
                  OR ingested_at IS NULL
                THEN 1 ELSE 0
            END
        ) AS rows_with_missing_provenance
    FROM `ftw-week-08`.`01_bronze`.`weather_raw`
    WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
)
SELECT
    source_name,
    actual_rows,
    expected_rows,
    rows_with_missing_provenance,
    CASE
        WHEN actual_rows = expected_rows
         AND rows_with_missing_provenance = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS current_status
FROM checks
ORDER BY source_name;


Bronze meets the core quality gate when the result contains three rows—`green_taxi`, `taxi_zones`, and `weather`—and each has `current_status = PASS`. The individual ingestion-log and same-file idempotency checks remain part of the completion evidence.